## Preprocess the dataset

In [16]:
# Import libraries
from datasets import load_dataset
from transformers import AutoProcessor, AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import wave
import numpy as np
import pandas as pd
import os
import re
import string

In [43]:
# Retrieve .txt files
def list_files_in_directory(directory):
    file_list = []
    for filename in os.listdir(directory):
        # Only pick up files with .txt extensions (transcript)
        if filename.endswith(".txt"):
            file_list.append(filename.replace(".txt", ""))
    return file_list

# Create the dataframe
def get_reference_df(directory, audio_txt_file):
    txt_file_path = os.path.join(directory, audio_txt_file + ".txt")
    columns = ["start_time", "end_time", "reference"]
    # Read the text file into a DataFrame
    df = pd.read_csv(txt_file_path, sep="\t", header=None, names=columns, quoting=3)

    # Add file name
    df.insert(0, 'file_name', pd.Series([audio_txt_file] * len(df)))

    # Remove quotation marks
    df['reference'] = df['reference'].apply(lambda x : x.replace('"',""))
    
    return df

# Trim audio files
def trim_wav_by_timestamps(directory, wav_file_name, reference_df):
    # Create the output directory if it doesn't exist
    output_dir = "data/sub/"
    os.makedirs(output_dir, exist_ok=True)
    wav_file = os.path.join(directory, wav_file_name + ".wav") # get into data file
    
    # Load the WAV file
    audio = AudioSegment.from_wav(wav_file)
    
    def trim_segments(row):
        start_ms = float(row['start_time']) * 1000  # Convert start time to milliseconds
        end_ms = float(row['end_time']) * 1000      # Convert end time to milliseconds
        trimmed_segment = audio[start_ms:end_ms]
    
        return trimmed_segment
    
    # Iterate over timestamps and trim the audio
    for i, row in reference_df.iterrows():
        trimmed_segment = trim_segments(row)
        output_file = os.path.join(output_dir, wav_file_name + "_" f"trimmed_segment_{i+1}.wav")
        trimmed_segment.export(output_file, format="wav")
        reference_df.at[i, 'trimmed_segment_path'] = output_file
    
    return reference_df

# Helper function to remove punctuations from original subtitles
def strip_punctuation(text):
    # Create a translation table that maps each punctuation character to None
    translator = str.maketrans('', '', string.punctuation)
    # Use the translation table to remove all punctuation from the text
    return text.translate(translator)

# Filter english text and append it to dataframe
def filter_subs_by_lang(reference_df):
    # Helper function that is applied across the rows to filter english text only
    
    def filter_english_only(text):
        # Define a regex pattern to match English words
        english_pattern = re.compile(r'\b[A-Za-z]+\b')
        # Find all English words in the text
        english_words = english_pattern.findall(text)
        # Join the English words into a single string
        english_text = ' '.join(english_words)
        return english_text
    
    def filter_thai_only(text):
        # Remove punctuation from text
        text = strip_punctuation(text)
        # Tokenize the string, split by spaces
        list_of_words = text.split()
        # Define a regex pattern to match English words
        english_pattern = re.compile(r'\b[A-Za-z]+\b')
        # Find all English words in the text
        english_words = english_pattern.findall(text)
        # Find all Thai words
        thai_words = [word for word in list_of_words if word not in english_words]
        # Concatenate the Thai words into a string
        thai_text = ' '.join(thai_words)
        # Return thai string
        return thai_text

    reference_df['eng_reference'] = reference_df['reference'].apply(filter_english_only)
    reference_df['thai_reference'] = reference_df['reference'].apply(filter_thai_only)

    return reference_df

def create_prompts(reference_df):
    prompt_template = "### USER:\n{human_prompt}\n\n### RESPONSE:\n"

    def create_full_prompts(text):
        prompt = "Translate the following text to English." + text
        full_prompt = prompt_template.format(human_prompt=prompt)
        return full_prompt

    reference_df['prompts'] = reference_df['thai_reference'].apply(create_full_prompts)
    return reference_df

def get_combined_audio_table(directory, file_names):
    combined_df = pd.DataFrame()
    for file_name in file_names:
        # Reads the transcript dataframe which has the start_time, end_time of each transcript
        reference_df = get_reference_df(directory, file_name)

        # Split subtitles by language
        reference_df = filter_subs_by_lang(reference_df)
        # Remove 'reference' column
        reference_df = reference_df.drop('reference', axis=1)

        # Create prompts for the model
        reference_df = create_prompts(reference_df)

        # Uncomment to trim all the .wav file according to the subtitles start_time and end_time
        ## reference_df = trim_wav_by_timestamps(directory, file_name, reference_df)
        
        # Append the processed DataFrame to the combined DataFrame
        combined_df = pd.concat([combined_df, reference_df], ignore_index=True)

        # Comment out this section if not required
        combined_df = combined_df.drop(['file_name', 'start_time', 'end_time'], axis=1)
    
    return combined_df

directory = os.path.join(os.getcwd(), "data/train/")
file_names = list_files_in_directory(directory)
combined_df = get_combined_audio_table(directory, file_names)
combined_df

,eng_reference,thai_reference,prompts
0,So today I will treat them to,เดี๋ยววันเนี่ยจะพาไปเลี้ยง,### USER:\nTranslate the following text to English.เดี๋ยววันเนี่ยจะพาไปเลี้ยง\n\n### RESPONSE:\n
1,spicy Tom Yum hot pot,ต้มยำหม้อไฟแซ่บๆ,### USER:\nTranslate the following text to English.ต้มยำหม้อไฟแซ่บๆ\n\n### RESPONSE:\n
2,Hello everyone This is,สวัสดีครับทุกคน เดือนนี้เป็นเดือน,### USER:\nTranslate the following text to English.สวัสดีครับทุกคน เดือนนี้เป็นเดือน\n\n### RESPONSE:\n
3,a month of giving out bonuses,เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ,### USER:\nTranslate the following text to English.เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ\n\n### RESPONSE:\n
4,And our company is,และบริษัทของเราเนี่ยเป็น,### USER:\nTranslate the following text to English.และบริษัทของเราเนี่ยเป็น\n\n### RESPONSE:\n
...,...,...,...
2408,Please like share and subscribe,ดูแล้วชอบ กดไลก์ แชร์ ซับสไครบ์ด้วย,### USER:\nTranslate the following text to English.ดูแล้วชอบ กดไลก์ แชร์ ซับสไครบ์ด้วย\n\n### RESPONSE:\n
2409,Chollada Channel Chollada Channel,กับ ค่ะ,### USER:\nTranslate the following text to English.กับ ค่ะ\n\n### RESPONSE:\n
2410,Bye,บาย,### USER:\nTranslate the following text to English.บาย\n\n### RESPONSE:\n
2411,Please like share and subscribe,อย่าลืมกดไลก์ กดแชร์ หรือว่าซับสไครบ์,### USER:\nTranslate the following text to English.อย่าลืมกดไลก์ กดแชร์ หรือว่าซับสไครบ์\n\n### RESPONSE:\n


## Import SEALION Model

In [27]:
generation_kwargs = {
    "do_sample": False,  # set to true if temperature is not 0
    "temperature": None,
    "max_new_tokens": 256,
    "top_k": 50,
    "top_p": 0.7,
    "repetition_penalty": 1.2,
}

In [28]:
MODEL_NAME = "aisingapore/sea-lion-7b"

# Import tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(
    "aisingapore/sealion7b-instruct-nc", 
    trust_remote_code=True,
    device_map="auto",
    low_cpu_mem_usage=True
)

generation_kwargs["eos_token_id"] = tokenizer.eos_token_id

In [12]:
# Remove unneeded kwargs
if generation_kwargs["do_sample"] == False:
    generation_kwargs.pop("temperature")
    generation_kwargs.pop("top_k")
    generation_kwargs.pop("top_p")

KeyError: 'temperature'

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    "aisingapore/sealion7b-instruct-nc", 
    trust_remote_code=True,
    device_map="auto",
    low_cpu_mem_usage=True
)

The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.84s/it]


In [44]:
# Generate tokens and generate output
def generate_translation(reference_df):

    def translate_thai(prompt):
        tokens = tokenizer(prompt, return_tensors="pt")
        output = model.generate(
            tokens["input_ids"],
            **generation_kwargs,
        )
        translation = tokenizer.decode(output[0], skip_special_tokens=True)
        translation = translation.partition("RESPONSE:\n")[2]
        return translation

    reference_df['translation'] = reference_df['prompts'].apply(translate_thai)
    return reference_df

combined_df = generate_translation(combined_df)
combined_df

/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/transformers/generation/utils.py:1659: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example 

In [2]:
# Import pre-trained model from HuggingFace

model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=MODEL_NAME, 
                                             trust_remote_code=True,
                                             device_map="auto",
                                             low_cpu_mem_usage=True)

The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.02s/it]


In [3]:
from transformers import pipeline

MODEL_NAME = "aisingapore/sea-lion-7b"

# device = 0 if torch.cuda.is_available() else "cpu"

pipe = pipeline(
    task="text-generation", 
    model=MODEL_NAME, 
    trust_remote_code=True,
    device_map="auto",
    low_cpu_mem_usage=True
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.07s/it]


In [ ]:
# Import LLM
llmTokenizer = AutoTokenizer.from_pretrained("aisingapore/sea-lion-7b", trust_remote_code=True)

# Preprocess the input data (Convert audio to tensors)
tokens = llmTokenizer.tokenize("Hello, who are you?", return_tensors="pt")
print(tokens)